<a href="https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/U03_library_curation_GTDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Library Curation — GTDB enrichment

Companion to the search notebook (`NB03_general`). It does **not** search or fetch proteomes; it
**annotates the shared taxon table** with the Genome Taxonomy Database (GTDB), which replaces
NCBI's name-based taxonomy with a genome-phylogeny-based, rank-normalized one and carries a CheckM
quality score for every genome. For each prokaryote in your library it adds: the GTDB lineage, the
CheckM completeness/contamination, and the GTDB **species-representative accession** (a
quality-controlled genome you can later fetch by accession instead of by TaxID).

It writes a **new** CSV alongside your current one — nothing is overwritten. GTDB covers **Bacteria
and Archaea only**, so your eukaryotes come through unmatched (expected). Run top to bottom once;
re-running is safe (the big metadata download is cached).

*Roadmap:* a later section here will **sample** GTDB representatives across bacterial/archaeal phyla
to broaden the library's phyletic coverage. This version only annotates what you already have.

## 0. Setup — mount Drive and locate the shared library

In [10]:
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=True)
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path

# ---- LOCATION ----
PROJECT_DRIVE = "/content/drive/MyDrive/BIOINFO4-5203-F26"

PROJECT  = Path(PROJECT_DRIVE)
DB_ROOT  = PROJECT / "Databases" / "proteome_library"   # your shared taxon table lives here
GTDB_DIR = PROJECT / "Databases" / "gtdb"               # cached GTDB metadata (a new shared resource)
for d in (DB_ROOT, GTDB_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("shared library:", DB_ROOT)
print("gtdb cache:    ", GTDB_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
shared library: /content/drive/MyDrive/BIOINFO4-5203-F26/Databases/proteome_library
gtdb cache:     /content/drive/MyDrive/BIOINFO4-5203-F26/Databases/gtdb


## 1. Configuration

`LIBRARY_CSV_IN` is your current taxon table (in `Databases/proteome_library/`). The enriched copy
is written to `LIBRARY_CSV_OUT` in the same folder. If a GTDB URL 404s, grab the current path from
https://gtdb.ecogenomic.org/downloads and paste it in — release numbers change.

In [16]:
LIBRARY_CSV_IN  = "Taxon_library_v1.csv"         # existing table in Databases/proteome_library/
LIBRARY_CSV_OUT = "Taxon_library_v1_gtdb.csv"    # enriched output (new file; does NOT overwrite)

GTDB_RELEASE = "232"                              # current GTDB release (11-RS232, Apr 2026)
_base = f"https://data.gtdb.ecogenomic.org/releases/release{GTDB_RELEASE}/{GTDB_RELEASE}.0"
GTDB_BAC_URL = f"{_base}/bac120_metadata_r{GTDB_RELEASE}.tsv.gz"   # if 404: see gtdb.../downloads
GTDB_AR_URL  = f"{_base}/ar53_metadata_r{GTDB_RELEASE}.tsv.gz"

MIN_COMPLETENESS_FLAG = 90.0                      # flag matched genomes below this CheckM completeness


## 2. Load the taxon table and pick its TaxID column

In [17]:
import pandas as pd

in_path = DB_ROOT / LIBRARY_CSV_IN
if not in_path.exists():
    raise FileNotFoundError(f"Taxon table not found at {in_path}")
lib = pd.read_csv(in_path)

def pick_taxid_col(df):
    for c in ("resolved_taxid", "NCBI_TaxID", "taxid", "species_taxid"):
        if c in df.columns:
            return c
    raise ValueError(f"No TaxID column in {list(df.columns)}")
TAXCOL = pick_taxid_col(lib)
print(f"{len(lib)} taxa loaded; TaxID column = '{TAXCOL}'")


149 taxa loaded; TaxID column = 'NCBI_TaxID'


## 3. Download GTDB metadata (cached)

Two files — Bacteria (`bac120`) and Archaea (`ar53`). A few hundred MB total; downloaded once to
`Databases/gtdb/` and skipped on re-run.

In [18]:
def fetch(url, dest):
    if dest.exists() and dest.stat().st_size > 0:
        print("cached:", dest.name); return
    print("downloading", dest.name, "...")
    rc = subprocess.run(["wget", "-q", "-O", str(dest), url]).returncode
    if rc != 0 or not dest.exists() or dest.stat().st_size == 0:
        if dest.exists(): dest.unlink()
        raise RuntimeError(f"download failed: {url}\n"
                           f"Check the current path at https://gtdb.ecogenomic.org/downloads "
                           f"and update the URL in Section 1.")
    print("  ->", dest.stat().st_size // (1024*1024), "MB")

bac_path = GTDB_DIR / f"bac120_metadata_r{GTDB_RELEASE}.tsv.gz"
ar_path  = GTDB_DIR / f"ar53_metadata_r{GTDB_RELEASE}.tsv.gz"
fetch(GTDB_BAC_URL, bac_path)
fetch(GTDB_AR_URL,  ar_path)


cached: bac120_metadata_r232.tsv.gz
cached: ar53_metadata_r232.tsv.gz


## 4. Load GTDB metadata and build lookups

Column names shift a little between releases (e.g. `checkm_` vs `checkm2_`), so the code detects the
columns it needs rather than hard-coding them. It keeps only the handful of columns required, so the
big files load in a few hundred MB of RAM.

In [19]:
def load_meta(path):
    hdr = pd.read_csv(path, sep="\t", nrows=0, compression="gzip").columns.tolist()
    def pick(names, contains=None):
        for n in names:
            if n in hdr: return n
        if contains:
            for c in hdr:
                if contains in c.lower(): return c
        return None
    cols = {
        "acc":     pick(["accession"]),
        "taxid":   pick(["ncbi_taxid"]),
        "gtdbtax": pick(["gtdb_taxonomy"]),
        "rep":     pick(["gtdb_representative"]),
        "comp":    pick(["checkm2_completeness", "checkm_completeness"], contains="completeness"),
        "cont":    pick(["checkm2_contamination", "checkm_contamination"], contains="contamination"),
        "name":    pick(["ncbi_organism_name"]),
        "gcat":    pick(["ncbi_genome_category"], contains="genome_category"),
    }
    use = [c for c in cols.values() if c]
    df = pd.read_csv(path, sep="\t", usecols=use, compression="gzip", dtype=str)
    ren = {v: k for k, v in cols.items() if v}          # rename to canonical short names
    return df.rename(columns=ren)

meta = pd.concat([load_meta(bac_path), load_meta(ar_path)], ignore_index=True)

meta["taxid_i"] = pd.to_numeric(meta.get("taxid"), errors="coerce")
meta = meta.dropna(subset=["taxid_i"]).copy()
meta["taxid_i"] = meta["taxid_i"].astype(int)
meta["isrep"]   = meta.get("rep", "").astype(str).str.lower().isin(["t", "true"])
meta["species"] = meta["gtdbtax"].str.extract(r"(s__[^;]*)")

# species -> its representative row; taxid -> best genome (representative preferred)
reps     = meta[meta["isrep"]].drop_duplicates("species").set_index("species")
by_taxid = meta.sort_values("isrep", ascending=False).drop_duplicates("taxid_i").set_index("taxid_i")
# optional name fallback (exact normalized NCBI organism name)
def _norm(s):
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())
if "name" in meta.columns:
    meta["name_norm"] = meta["name"].map(_norm)
    by_name = meta.sort_values("isrep", ascending=False).drop_duplicates("name_norm").set_index("name_norm")
else:
    by_name = None
print(f"GTDB metadata: {len(meta):,} genomes, {len(reps):,} species representatives")


GTDB metadata: 901,341 genomes, 199,923 species representatives


## 5. Enrich the table and write the new CSV

For each taxon: match on NCBI TaxID (then, if needed, on exact NCBI name), resolve to the GTDB
**species representative**, and attach its lineage, quality, and accession. Unmatched rows (all
eukaryotes, plus any prokaryote GTDB can't place by these keys) keep blank GTDB columns.

In [20]:
def _strip_prefix(a):
    a = str(a)
    return a[3:] if a[:3] in ("RS_", "GB_") else a   # RS_GCF_000005845.2 -> GCF_000005845.2

def _rep_for(row):
    sp = row.get("species")
    r = reps.loc[sp].to_dict() if (sp is not None and sp in reps.index) else row.to_dict()
    return r

records = []
for _, lr in lib.iterrows():
    out = {"gtdb_taxonomy": "", "accession": "", "checkm_completeness": "",
           "checkm_contamination": "", "gtdb_match": "none"}
    try:
        tx = int(float(lr[TAXCOL]))
    except (TypeError, ValueError):
        tx = None
    row = None; how = "none"
    if tx is not None and tx in by_taxid.index:
        row = by_taxid.loc[tx]; how = "taxid"
    elif by_name is not None:
        nm = _norm(lr.get("Organism", lr.get("organism", "")))
        if nm and nm in by_name.index:
            row = by_name.loc[nm]; how = "name"
    if row is not None:
        rep = _rep_for(row)
        out.update(gtdb_taxonomy=rep.get("gtdbtax", ""),
                   accession=_strip_prefix(rep.get("acc", "")),
                   checkm_completeness=rep.get("comp", ""),
                   checkm_contamination=rep.get("cont", ""),
                   gtdb_match=how)
    records.append(out)

enr = pd.concat([lib.reset_index(drop=True), pd.DataFrame(records)], axis=1)
out_path = DB_ROOT / LIBRARY_CSV_OUT
enr.to_csv(out_path, index=False)

matched = (enr["gtdb_match"] != "none").sum()
print(f"GTDB match: {matched}/{len(enr)} taxa  ({(enr['gtdb_match']=='taxid').sum()} by TaxID, "
      f"{(enr['gtdb_match']=='name').sum()} by name)")
comp = pd.to_numeric(enr["checkm_completeness"], errors="coerce")
low = enr[(enr["gtdb_match"] != "none") & (comp < MIN_COMPLETENESS_FLAG)]
if len(low):
    print(f"\nMatched but CheckM completeness < {MIN_COMPLETENESS_FLAG}% (watch for false 'absences'):")
    for _, r in low.iterrows():
        print(f"  - {r.get('Organism','?')}: {r['checkm_completeness']}% complete")
unm = enr[enr["gtdb_match"] == "none"]
print(f"\nUnmatched ({len(unm)}) — eukaryotes are expected here:")
for _, r in unm.iterrows():
    print(f"  - {r.get('Organism','?')} (TaxID {r.get(TAXCOL,'?')})")
print(f"\nWritten: {out_path}")


GTDB match: 121/149 taxa  (112 by TaxID, 9 by name)

Matched but CheckM completeness < 90.0% (watch for false 'absences'):
  - Escherichia coli K-12 subMG1655: 42.86% complete
  - Prochlorococcus marinus MIT 9215: 85.56% complete
  - Nitrospira defluvii: 66.81% complete

Unmatched (28) — eukaryotes are expected here:
  - Yarrowia lipolytica (TaxID 4952.0)
  - Tetrahymena thermophila (TaxID 312017.0)
  - Homo sapiens (TaxID 9606.0)
  - Bos taurus (TaxID 9913.0)
  - Drosophila melanogaster (TaxID 7227.0)
  - Mus musculus (TaxID 10090.0)
  - Ovis aries (TaxID 9940.0)
  - Chroococcidiopsis sp. PCC 6712 (TaxID 118675.0)
  - Cyanothece sp. BH68, ATCC 51142 (TaxID 2546360.0)
  - Cyanothece sp. PCC 7425 (TaxID 395961.0)
  - Amborella trichopoda (TaxID 13333.0)
  - Nicotiana tabacum (TaxID 4097.0)
  - Spinacia oleracea (TaxID 3562.0)
  - Arabidopsis thaliana (TaxID 3702.0)
  - Marchantia polymorpha (TaxID 3197.0)
  - Physcomitrium patens (TaxID 3218.0)
  - Selaginella moellendorffii (TaxID 8803

## 6. Broaden coverage — sample GTDB representatives (isolates preferred)

Adds new taxa to fill thin parts of the tree, **without touching your curated ones**. One
quality-controlled **isolate** representative per phylum, plus extra depth in the bioenergetically
rich phyla, spread across classes. Reads your enriched table to see what you already cover, tops up
toward the targets, appends the new taxa (tagged `gtdb_sampled`, with accessions), and writes a new
`Taxon_library_v2.csv`. Additive and non-destructive — re-run with different settings to regenerate.

In [21]:
# ---- sampling settings ----
LIBRARY_ENRICHED    = LIBRARY_CSV_OUT        # your enriched table (from section 5)
LIBRARY_SAMPLED_OUT = "Taxon_library_v2.csv" # new file with the appended taxa

TARGET_RICH  = 3     # representatives per bioenergetically rich phylum
TARGET_OTHER = 1     # representatives per other phylum (tree-of-life backbone)
SAMPLE_MIN_COMPLETENESS = 90.0   # CheckM completeness floor
SAMPLE_MAX_CONTAMINATION = 5.0   # CheckM contamination ceiling
PREFER_ISOLATES   = True         # take isolates before MAGs/SAGs
ALLOW_MAG_FALLBACK = False        # if a phylum has no qualifying isolate, skip it (keep the library clean)

# phyla to sample more deeply (GTDB names; alternates included so spelling shifts still match)
RICH_PHYLA = {
    "pseudomonadota", "proteobacteria", "actinomycetota", "actinobacteriota",
    "bacillota", "firmicutes", "bacteroidota", "campylobacterota", "nitrospirota",
    "aquificota", "deinococcota", "chloroflexota", "desulfobacterota",
    "thermodesulfobacteriota", "halobacteriota", "methanobacteriota", "thermoproteota",
}


## 7. Run the sampling and write v2

In [22]:
import re

def _tax(s, rank):   # pull one rank (e.g. 'p') out of a gtdb lineage string
    m = re.search(rf"{rank}__([^;]*)", str(s))
    return m.group(1).strip() if m else ""

# candidate pool = species representatives passing quality, with derived fields
pool = meta[meta["isrep"]].copy()
pool["phylum"]  = pool["gtdbtax"].map(lambda s: _tax(s, "p"))
pool["klass"]   = pool["gtdbtax"].map(lambda s: _tax(s, "c"))
pool["domain"]  = pool["gtdbtax"].map(lambda s: _tax(s, "d"))
pool["species"] = pool["gtdbtax"].map(lambda s: _tax(s, "s"))
pool["comp_f"]  = pd.to_numeric(pool.get("comp"), errors="coerce")
pool["cont_f"]  = pd.to_numeric(pool.get("cont"), errors="coerce")
if "gcat" in pool.columns:
    _gc = pool["gcat"].astype(str).str.lower()
    pool["is_isolate"] = ~_gc.str.contains("metagenome|single cell|environmental", na=False)
else:
    print("NOTE: no genome-category column found; treating all as isolates.")
    pool["is_isolate"] = True
pool = pool[(pool["comp_f"] >= SAMPLE_MIN_COMPLETENESS) &
            (pool["cont_f"] <= SAMPLE_MAX_CONTAMINATION) &
            (pool["phylum"] != "")].copy()

# what you already cover, from the enriched table
enr = pd.read_csv(DB_ROOT / LIBRARY_ENRICHED)
enr_phylum = enr["gtdb_taxonomy"].map(lambda s: _tax(s, "p")) if "gtdb_taxonomy" in enr else pd.Series([], dtype=str)
have = enr_phylum[enr_phylum != ""].value_counts().to_dict()
existing_species = set(enr["gtdb_taxonomy"].map(lambda s: _tax(s, "s"))) if "gtdb_taxonomy" in enr else set()
existing_acc = set(enr["accession"].astype(str)) if "accession" in enr else set()

def target_for(phylum):
    return TARGET_RICH if phylum.lower() in RICH_PHYLA else TARGET_OTHER

def pick_from_phylum(cands, need):
    """isolates first; spread across classes; then by completeness."""
    iso = cands[cands["is_isolate"]]
    if PREFER_ISOLATES:
        base = iso
        if len(base) < need and ALLOW_MAG_FALLBACK:            # top up with MAGs only if allowed
            base = pd.concat([iso, cands[~cands.index.isin(iso.index)]])
    else:
        base = cands
    if len(base) == 0:                                         # no qualifying genome -> skip phylum
        return []
    base = base.sort_values(["is_isolate", "comp_f"], ascending=[False, False])
    picked, used = [], set()
    for _, r in base.iterrows():                       # one per class first (spread)
        if len(picked) >= need: break
        if r["klass"] in used: continue
        picked.append(r); used.add(r["klass"])
    for _, r in base.iterrows():                        # then fill ignoring class
        if len(picked) >= need: break
        if any(r["acc"] == p["acc"] for p in picked): continue
        picked.append(r)
    return picked

new_rows, added_by_phylum = [], {}
for phylum, grp in pool.groupby("phylum"):
    need = max(0, target_for(phylum) - have.get(phylum, 0))
    if need == 0:
        continue
    grp = grp[~grp["species"].isin(existing_species) & ~grp["acc"].map(_strip_prefix).isin(existing_acc)]
    if len(grp) == 0:
        continue
    for r in pick_from_phylum(grp, need):
        acc = _strip_prefix(r["acc"])
        if acc in existing_acc:
            continue
        existing_acc.add(acc)
        new_rows.append({
            "Tier": "8 - GTDB backbone", "Category": "GTDB phylum coverage",
            "Organism": r.get("name", "") or r["species"].replace("s__", ""),
            "Domain": r["domain"], "Clade_or_Phylum": r["phylum"],
            "NCBI_TaxID": r.get("taxid", ""), "TaxID_status": "gtdb_rep",
            "CI_form": "", "Rationale": f"GTDB isolate representative ({r['phylum']}) coverage top-up",
            "Source": "gtdb_sampled", "gtdb_taxonomy": r["gtdbtax"], "accession": acc,
            "checkm_completeness": r.get("comp", ""), "checkm_contamination": r.get("cont", ""),
            "gtdb_match": "gtdb_rep",
        })
        added_by_phylum[phylum] = added_by_phylum.get(phylum, 0) + 1

# append, aligning columns to the enriched table
new_df = pd.DataFrame(new_rows)
for c in enr.columns:
    if c not in new_df.columns: new_df[c] = ""
if len(new_df):
    new_df = new_df[enr.columns]
combined = pd.concat([enr, new_df], ignore_index=True)
out_path = DB_ROOT / LIBRARY_SAMPLED_OUT
combined.to_csv(out_path, index=False)

# report: before -> after per phylum
print(f"Added {len(new_rows)} taxa across {len(added_by_phylum)} phyla; "
      f"library {len(enr)} -> {len(combined)}.\n")
after = have.copy()
for ph, n in added_by_phylum.items():
    after[ph] = after.get(ph, 0) + n
print(f"{'phylum':32s} before  after")
for ph in sorted(set(list(have) + list(added_by_phylum)), key=lambda p: -after.get(p, 0)):
    tag = "  <- rich" if ph.lower() in RICH_PHYLA else ""
    grew = "  +" + str(added_by_phylum[ph]) if ph in added_by_phylum else ""
    print(f"{ph[:32]:32s} {have.get(ph,0):5d}  {after.get(ph,0):5d}{grew}{tag}")
print(f"\nWritten: {out_path}")


Added 60 taxa across 54 phyla; library 149 -> 209.

phylum                           before  after
Cyanobacteriota                     82     82
Pseudomonadota                      13     13  <- rich
Chloroflexota                        5      5  <- rich
Desulfobacterota                     1      3  +2  <- rich
Actinomycetota                       1      3  +2  <- rich
Bacillota                            1      3  +2  <- rich
Nitrospirota                         2      3  +1  <- rich
Campylobacterota                     2      3  +1  <- rich
Bacteroidota                         2      3  +1  <- rich
Aquificota                           1      3  +2  <- rich
Thermoproteota                       2      3  +1  <- rich
Deinococcota                         2      3  +1  <- rich
Methanobacteriota                    0      3  +3  <- rich
Halobacteriota                       2      3  +1  <- rich
Verrucomicrobiota                    2      2
Zhuqueibacterota                     0      1  +1


## 8. What you now have, and what's next

The enriched table adds four columns — `gtdb_taxonomy`, `accession` (the GTDB representative genome),
`checkm_completeness`, `checkm_contamination`, `gtdb_match` — while keeping every original column, so
`NB03_general` still reads it unchanged (it just ignores the extras). Two payoffs: the `gtdb_taxonomy`
column gives every prokaryote an authoritative, rank-consistent lineage (reconciling NCBI's renamed
genera), and the `checkm_completeness` column tells you which proteomes are complete enough to trust
a `no_hits` result.

*Next step, when you want it:* teach `NB03_general` to fetch **by accession** (using the `accession`
column now present on both matched and GTDB-sampled rows) so the exact GTDB representative genome is
retrieved — this removes strain-vs-species ambiguity entirely and pins the new backbone taxa to their
quality-controlled assemblies. Until then, `NB03_general` reads `Taxon_library_v2.csv` unchanged and
fetches the sampled taxa by their NCBI TaxID like the rest.